In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# MNIST dataset with torchvision datasets and transforms
train_dataset = datasets.MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor())

# Data loader configuration
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# Define the MLP model architecture
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.layers(x)

# Initialize the model and optimizer
model = MLP().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

100%|██████████| 9912422/9912422 [00:00<00:00, 161557575.86it/s]

Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw


100%|██████████| 28881/28881 [00:00<00:00, 77254906.78it/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw



100%|██████████| 1648877/1648877 [00:00<00:00, 47932157.86it/s]

Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw


100%|██████████| 4542/4542 [00:00<00:00, 3671329.50it/s]


Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



### Part 1: Train the MLP and Evaluate on Clean and Attacked Test Sets

In [ ]:
# Training function for the MLP
def train(model, device, train_loader, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            optimizer.step()

# Testing function to evaluate model accuracy
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset)
    print(f'Test on clean set - Loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({100. * correct / len(test_loader.dataset):.2f}%)')

# FGSM attack function
def fgsm_attack(image, epsilon, data_grad):
    sign_data_grad = data_grad.sign()
    perturbed_image = image + epsilon * sign_data_grad
    perturbed_image = torch.clamp(perturbed_image, 0, 1)
    return perturbed_image

# Training the model
train(model, device, train_loader, optimizer)

# Testing on clean test set
print("Testing on clean test set:")
test(model, device, test_loader)

def test_attack(model, device, test_loader, epsilon=0.1):
    correct = 0
    model.eval()
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        data.requires_grad = True
        output = model(data)
        init_pred = output.max(1, keepdim=True)[1]

        if not init_pred.eq(target.view_as(init_pred)).all():
            continue

        loss = F.cross_entropy(output, target)
        model.zero_grad()
        loss.backward()
        data_grad = data.grad.data
        perturbed_data = fgsm_attack(data, epsilon, data_grad)
        output = model(perturbed_data)
        final_pred = output.max(1, keepdim=True)[1]
        correct += final_pred.eq(target.view_as(final_pred)).sum().item()

    final_acc = correct / float(len(test_loader.dataset))
    print(f'Test on FGSM attacked set (Epsilon: {epsilon}) - Accuracy: {correct}/{len(test_loader.dataset)} ({final_acc:.4f})')

print("Testing on FGSM attacked test set:")
test_attack(model, device, test_loader)

Testing on clean test set:
Test on clean set - Loss: 0.0943, Accuracy: 9768/10000 (97.68%)
Testing on FGSM attacked test set:
Test on FGSM attacked set (Epsilon: 0.1) - Accuracy: 908/10000 (0.0908)


### Part 2: Adversarial Training Using FGSM

In [ ]:
# Function to perform adversarial training using FGSM
def adversarial_train(model, device, train_loader, optimizer, epsilon=0.1, epochs=10):
    model.train()
    for epoch in range(epochs):
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            data.requires_grad = True
            output = model(data)
            loss = F.cross_entropy(output, target)
            optimizer.zero_grad()
            loss.backward()
            data_grad = data.grad.data
            perturbed_data = fgsm_attack(data, epsilon, data_grad)
            output_adv = model(perturbed_data)
            loss_adv = F.cross_entropy(output_adv, target)
            optimizer.zero_grad()
            loss_adv.backward()
            optimizer.step()
        print(f'Adversarial Training - Epoch: {epoch+1}/{epochs}, Loss: {loss_adv.item():.4f}')

# Adversarially train the MLP
model_adv = MLP().to(device)
optimizer_adv = optim.Adam(model_adv.parameters(), lr=0.001)

adversarial_train(model_adv, device, train_loader, optimizer_adv)

# Evaluate the adversarially trained model on clean data
test(model_adv, device, test_loader)

# Evaluate the adversarially trained model on FGSM attacked data
print("Testing adversarially trained model on FGSM attacked test set:")
test_attack(model_adv, device, test_loader)

Adversarial Training - Epoch: 1/10, Loss: 0.5718
Adversarial Training - Epoch: 2/10, Loss: 0.5342
Adversarial Training - Epoch: 3/10, Loss: 0.7460
Adversarial Training - Epoch: 4/10, Loss: 0.5364
Adversarial Training - Epoch: 5/10, Loss: 0.2169
Adversarial Training - Epoch: 6/10, Loss: 0.2353
Adversarial Training - Epoch: 7/10, Loss: 0.1869
Adversarial Training - Epoch: 8/10, Loss: 0.4022
Adversarial Training - Epoch: 9/10, Loss: 0.2383
Adversarial Training - Epoch: 10/10, Loss: 0.7363
Test on clean set - Loss: 0.0588, Accuracy: 9817/10000 (98.17%)
Testing adversarially trained model on FGSM attacked test set:
Test on FGSM attacked set (Epsilon: 0.1) - Accuracy: 3968/10000 (0.3968)


#### Adversarial Training Using FGSM with varying values of Epsilon to improve accuracy

In [ ]:
def adversarial_train_varied_epsilon(model, device, train_loader, optimizer, epochs=10, epsilon_max=0.1):
    model.train()
    for epoch in range(epochs):
        for batch_idx, (data, target) in enumerate(train_loader):
            # Move data to the appropriate device
            data, target = data.to(device), target.to(device)

            # Set requires_grad to True for input data to calculate gradient with respect to input
            data.requires_grad = True

            # Forward pass
            output = model(data)
            loss = F.cross_entropy(output, target)

            # Calculate gradients
            optimizer.zero_grad()
            loss.backward()

            # Generate adversarial example using the gradients
            data_grad = data.grad.data
            # Randomly choose epsilon for the current batch
            epsilon = np.random.uniform(low=0.0, high=epsilon_max)
            perturbed_data = fgsm_attack(data, epsilon, data_grad)

            # Perform a forward pass with the adversarial example
            output_adv = model(perturbed_data)
            loss_adv = F.cross_entropy(output_adv, target)

            # Backward pass and optimize with the adversarial examples
            optimizer.zero_grad()
            loss_adv.backward()
            optimizer.step()

            if batch_idx % 100 == 0:
                print(f'Adversarial Training - Epoch: {epoch+1}/{epochs}, '
                      f'Batch: {batch_idx}/{len(train_loader)}, '
                      f'Loss: {loss_adv.item():.4f}, Epsilon: {epsilon:.4f}')

model_adv = MLP().to(device)
optimizer_adv = optim.Adam(model_adv.parameters(), lr=0.001)

# Perform the adversarial training with varied epsilon
adversarial_train_varied_epsilon(model_adv, device, train_loader, optimizer_adv)

# Evaluate the adversarially trained model on clean data
print("Evaluating on clean test set:")
test(model_adv, device, test_loader)

# Evaluate the adversarially trained model on FGSM attacked data
print("Evaluating on FGSM attacked test set:")
test_attack(model_adv, device, test_loader, epsilon=0.1)

Adversarial Training - Epoch: 1/10, Batch: 0/938, Loss: 2.3339, Epsilon: 0.0178
Adversarial Training - Epoch: 1/10, Batch: 100/938, Loss: 0.8737, Epsilon: 0.0151
Adversarial Training - Epoch: 1/10, Batch: 200/938, Loss: 0.4987, Epsilon: 0.0173
Adversarial Training - Epoch: 1/10, Batch: 300/938, Loss: 0.3962, Epsilon: 0.0243
Adversarial Training - Epoch: 1/10, Batch: 400/938, Loss: 0.5820, Epsilon: 0.0571
Adversarial Training - Epoch: 1/10, Batch: 500/938, Loss: 0.2723, Epsilon: 0.0158
Adversarial Training - Epoch: 1/10, Batch: 600/938, Loss: 0.3571, Epsilon: 0.0590
Adversarial Training - Epoch: 1/10, Batch: 700/938, Loss: 0.2626, Epsilon: 0.0346
Adversarial Training - Epoch: 1/10, Batch: 800/938, Loss: 0.3351, Epsilon: 0.0538
Adversarial Training - Epoch: 1/10, Batch: 900/938, Loss: 0.2190, Epsilon: 0.0488
Adversarial Training - Epoch: 2/10, Batch: 0/938, Loss: 0.3280, Epsilon: 0.0663
Adversarial Training - Epoch: 2/10, Batch: 100/938, Loss: 0.4451, Epsilon: 0.0740
Adversarial Training